In [3]:
import matplotlib.pyplot as plt
import os
import sys
import cv2
import numpy as np
import subprocess
from pathlib import Path
from tqdm import tqdm
import torchvision.transforms as T
from PIL import Image
import preprocess_bg
import preprocess_product

In [4]:
def pad_to_square(image):
    w, h = image.size
    max_wh = max(w, h)
    hp = (max_wh - w) // 2
    vp = (max_wh - h) // 2
    # padding is (left, top, right, bottom)
    padding = (hp, vp, max_wh - w - hp, max_wh - h - vp)
    return T.functional.pad(image, padding, 0, 'constant')

base_prep = T.Compose([
    T.Lambda(pad_to_square),
    T.Resize((448, 448)) # Matching Inference resolution
])

try:
    from pycocotools.coco import COCO
except ImportError:
    print("Installing pycocotools...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pycocotools"])
    from pycocotools.coco import COCO

IMAGE_DIR = "/d2s_images_v1/images"

ANNOTATION_FILE = "./annotations/D2S_train_80.json"
# ANNOTATION_FILE = "./annotations/D2S_val_10.json"
# ANNOTATION_FILE = "./annotations/D2S_test_10.json"

OUTPUT_DIR = "./training_data"
# OUTPUT_DIR = "./validation_data"
# OUTPUT_DIR = "./testing_data"

IMAGE_RANDOM_DUPLICATES = 2
# ---------------------
def create_masked_dataset():
    batch_id = 0
    print(f"Loading annotations from {ANNOTATION_FILE}...")
    try:
        coco = COCO(ANNOTATION_FILE)
    except Exception as e:
        print(f"CRITICAL ERROR: Could not load COCO object. {e}")
        return
    Path(OUTPUT_DIR).mkdir(exist_ok=True)
    img_ids = coco.getImgIds()
    print(f"Found {len(img_ids)} images. Starting processing...")

    for img_id in tqdm(img_ids):
        img_info = coco.loadImgs(img_id)[0]
        file_name = img_info['file_name']
        img_path = os.path.join(IMAGE_DIR, file_name)

        original_img = cv2.imread(img_path)
        if original_img is None:
            continue

        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        for i, ann in enumerate(anns):
            for duplicate_id in range(IMAGE_RANDOM_DUPLICATES): # Duplicate unique product
                # Masking (Removes background)
                mask = coco.annToMask(ann)
                masked_img = cv2.bitwise_and(original_img, original_img, mask=mask)

                # Crop to Bounding Box
                x, y, w, h = [int(val) for val in ann['bbox']]
                h_img, w_img = original_img.shape[:2]
                y_min, y_max = max(0, y), min(h_img, y + h)
                x_min, x_max = max(0, x), min(w_img, x + w)
                crop_bgr = masked_img[y_min:y_max, x_min:x_max]
                if crop_bgr.size == 0: continue
                rotated_crop = preprocess_bg.apply_camera_roll_single(crop_bgr, max_angle=4)

                # Convert to PIL for the Square Pad
                crop_rgb = cv2.cvtColor(rotated_crop, cv2.COLOR_BGR2RGB)
                crop_pil = Image.fromarray(crop_rgb)

                # Apply Padding and Resize
                final_image_pil = base_prep(crop_pil)

                # Convert back to BGR for OpenCV saving
                final_save_img = cv2.cvtColor(np.array(final_image_pil), cv2.COLOR_RGB2BGR)
                generated_shelf = preprocess_bg.generate_bg_shelve(set_height=1024, set_width=2048)
                final_save_img = preprocess_product.paste_single_product_to_shelf(final_save_img, generated_shelf)
                final_blurry_img = preprocess_product.paste_single_product_to_shelf(final_save_img, generated_shelf, blur_foreground=True)

                # Get Category Name & Save
                cat_id = ann['category_id']
                cats = coco.loadCats(cat_id)
                cat_name = cats[0]['name'] if cats else "unknown"

                save_folder = Path(OUTPUT_DIR) / cat_name
                save_folder.mkdir(parents=True, exist_ok=True)

                # Clean filename to prevent overwrites
                clean_name = Path(file_name).stem
                save_filename = f"synthetic_{batch_id:06d}_{duplicate_id}_clean.png"

                # Focal Blur
                blurry_filename = f"synthetic_{batch_id:06d}_{duplicate_id}_blurry.png"

                # Save and Debug
                if final_blurry_img is not None and final_blurry_img.size > 0:
                    cv2.imwrite(str(save_folder / blurry_filename), final_blurry_img)

                if final_save_img is not None and final_save_img.size > 0:
                    cv2.imwrite(str(save_folder / save_filename), final_save_img)

                    # Optional: Show plot only for the first few to save time
                    # plt.imshow(cv2.cvtColor(final_save_img, cv2.COLOR_BGR2RGB))
                    # plt.show()
            batch_id += 1
        # break # prevent loop for now
    print(f"Done! Processed images saved to: {os.path.abspath(OUTPUT_DIR)}")

if __name__ == "__main__":
    create_masked_dataset()

Loading annotations from ./annotations/D2S_test_10.json...
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Found 438 images. Starting processing...


100%|██████████| 438/438 [03:04<00:00,  2.38it/s]

Done! Processed images saved to: D:\Users\Thomas Work\Desktop\Dataset Processing Practice2\testing_data
